In [3]:
import util.data_loading
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score
import numpy as np
import duckdb

from sklearn.metrics import classification_report

### Professor Feedback
Instead of predicting price, predict % change in price from previous year. But instead of numeric value you can turn it into a classification problem by splitting the % changes into ranges (like 0%-1%, 1%-2%, 2%-3%, etc.). Then see if you can use classification (decision tree / random forest, SVM, logistic regression, etc) to get a good prediction about how big the changes in price will be.

In [24]:
# debt_df= util.data_loading.load_raw_data('debt_2003_2025_clean.csv')
# debt_df2= util.data_loading.load_raw_data('debt_pre_2003.csv')
#
# debt_df2= util.data_loading.load_raw_data('debt_pre_2003.csv')
# debt_df2 = debt_df2.T
#
# debt_df2.columns = debt_df2.iloc[0]
# debt_df2 = debt_df2[1:]
#
# debt_df2 = debt_df2.reset_index()
# debt_df2 = debt_df2.rename(columns={"index": "quarter"})
#
# debt_df = util.data_loading.clean_dates(debt_df)
# debt_df2 = util.data_loading.clean_dates(debt_df2)
#
# debt_all = pd.concat(
#     [debt_df2, debt_df],
#     axis=0,          # stack rows
#     ignore_index=True
# )
#
# units_all =  util.data_loading.load_raw_data('housing_units_all.csv')
#
# median = util.data_loading.load_raw_data('MedianPricesofExistingDetachedHomesHistoricalData - Median Price.csv')
# median_all  = util.data_loading.clean_median(median)
#
# population = util.data_loading.load_raw_data('population_clean.csv')



Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/debt_2003_2025 _clean.csv
Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/debt_pre_2003.csv
Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/debt_pre_2003.csv
Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/housing_units_all.csv
Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/MedianPricesofExistingDetachedHomesHistoricalData - Median Price.csv
Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv


In [4]:
connection = duckdb.connect(database='housing.duckdb', read_only=True)


In [5]:
debt_all = connection.execute("""
SELECT
    *
FROM all_debt
WHERE CAST('20' || SUBSTR(quarter, 1, 2) AS INT) BETWEEN 1999 AND 2025;
""").fetch_df()

median_all = connection.execute("""
SELECT *
FROM median_price;
""").fetchdf()
median_all  = util.data_loading.clean_median(median_all)

units_all = connection.execute("""
SELECT *
FROM housing_units;
""").fetchdf()

population = connection.execute("""
SELECT *
FROM population;
""").fetchdf()



In [8]:
def logistic_regression(city, debt_type ="Mortgage"):

    # Median Price specific area
    median = util.data_loading.get_median_prices(median_all,city)
    median = median[(median["year"] >= 1999) & (median["year"] <= 2024)]
    median["median_pct_change"] = median[city].pct_change()

    # Unit Estimates specific area
    units = util.data_loading.get_unit_estimates(units_all, city + " County")
    units = units[(units["year"] >= 1999) & (units["year"] <= 2024)]
    units["units_pct_change"] = units["units"].pct_change()

    #Debt
    debt =  util.data_loading.get_debt(debt_all,debt_type)
    debt = debt[(debt["year"] >= 1999) & (debt["year"] <= 2024)]
    debt = debt.apply(pd.to_numeric, errors="coerce")
    debt[debt_type+"_pct_change"] = debt[debt_type].pct_change()


    # Population national

    population["pop_pct_change"] = population["population"].pct_change()

    df = (
    median.merge(debt, on="year", how="inner")
       .merge(units, on="year", how="inner")
    .merge(population, on="year", how="inner")
    )
    # Lag features
    df["median_pct_change_lag1"] = df["median_pct_change"].shift(1)
    df["Mortgage_change_lag1"]   = df["Mortgage_pct_change"].shift(1)
    df["units_change_lag1"]      = df["units_pct_change"].shift(1)
    df["pop_change_lag1"]        = df["pop_pct_change"].shift(1)

    df["median_pct_change_lag2"] = df["median_pct_change"].shift(2)
    df["Mortgage_change_lag2"]   = df["Mortgage_pct_change"].shift(2)
    df["units_change_lag2"]      = df["units_pct_change"].shift(2)
    df["pop_change_lag2"]        = df["pop_pct_change"].shift(2)


    df = df.fillna(0)
    df["rolling_pct_avg3"] = df["median_pct_change"].rolling(3).mean().shift(1)
    df = df.fillna(0)


    # threshold = df["median_pct_change"].median()
    bins = [-1.0, .09, 1.0]
    labels = [0, 1]

    df["price_change_class"] = pd.cut(
        df["median_pct_change"],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    f = [
    "Mortgage","Mortgage_pct_change", "Mortgage_change_lag1","Mortgage_change_lag2",
    "units","units_pct_change", "units_change_lag1","units_change_lag2",
    "population", "pop_pct_change", "pop_change_lag1","pop_change_lag2",
    "median_pct_change_lag1","median_pct_change_lag2",
    "rolling_pct_avg3"
    ]

    X = df[f]
    y = df["price_change_class"]


    log_reg_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            solver="lbfgs",
            class_weight="balanced",
            max_iter=1000,
            random_state=0
        )
    )


    tscv = TimeSeriesSplit(n_splits=3)
    accuracies = []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        if y_train.nunique() < 2:
            print(f"Skipping fold {fold} due to only one class being present.")
            continue

        log_reg_model.fit(X_train, y_train)
        y_pred = log_reg_model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        accuracies.append(acc)

        print(f"Fold {fold} Accuracy: {acc:.3f}")
        print(classification_report(y_test, y_pred, zero_division=0))


    print("Cross-val accuracy:", np.mean(accuracies))

    return df



In [9]:
d = logistic_regression("Los Angeles")

Skipping fold 0 due to only one class being present.
Fold 1 Accuracy: 0.500
              precision    recall  f1-score   support

           0       0.60      0.75      0.67         4
           1       0.00      0.00      0.00         2

    accuracy                           0.50         6
   macro avg       0.30      0.38      0.33         6
weighted avg       0.40      0.50      0.44         6

Fold 2 Accuracy: 0.667
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         4
           1       0.00      0.00      0.00         2

    accuracy                           0.67         6
   macro avg       0.33      0.50      0.40         6
weighted avg       0.44      0.67      0.53         6

Cross-val accuracy: 0.5833333333333333


In [10]:
logistic_regression("San Diego")


Fold 0 Accuracy: 0.667
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         4
           1       0.00      0.00      0.00         2

    accuracy                           0.67         6
   macro avg       0.33      0.50      0.40         6
weighted avg       0.44      0.67      0.53         6

Fold 1 Accuracy: 0.833
              precision    recall  f1-score   support

           0       0.83      1.00      0.91         5
           1       0.00      0.00      0.00         1

    accuracy                           0.83         6
   macro avg       0.42      0.50      0.45         6
weighted avg       0.69      0.83      0.76         6

Fold 2 Accuracy: 0.667
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         4
           1       0.00      0.00      0.00         2

    accuracy                           0.67         6
   macro avg       0.33      0.50      0.40         6
weigh

,San Diego,year,median_pct_change,Mortgage,Mortgage_pct_change,units,units_pct_change,population,pop_pct_change,median_pct_change_lag1,Mortgage_change_lag1,units_change_lag1,pop_change_lag1,median_pct_change_lag2,Mortgage_change_lag2,units_change_lag2,pop_change_lag2,rolling_pct_avg3,price_change_class
0,278906.0,2000,0.168119,3.90,0.157270,1044144,0.000000,33987.977,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1
1,311835.0,2001,0.118065,4.15,0.064103,1060146,0.015325,34479.458,0.014460,0.168119,0.157270,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1
2,381440.0,2002,0.223211,4.66,0.122892,1075986,0.014941,34871.843,0.011380,0.118065,0.064103,0.015325,0.014460,0.168119,0.157270,0.000000,0.000000,0.000000,1
3,466030.0,2003,0.221765,5.66,0.214592,1090165,0.013178,35253.159,0.010935,0.223211,0.122892,0.014941,0.011380,0.118065,0.064103,0.015325,0.014460,0.169798,1
4,577041.0,2004,0.238206,6.36,0.123675,1108796,0.017090,35574.576,0.009117,0.221765,0.214592,0.013178,0.010935,0.223211,0.122892,0.014941,0.011380,0.187680,1
5,603679.0,2005,0.046163,7.10,0.116352,1125143,0.014743,35827.943,0.007122,0.238206,0.123675,0.017090,0.009117,0.221765,0.214592,0.013178,0.010935,0.227727,0
6,585974.0,2006,-0.029329,8.23,0.159155,1140067,0.013264,36021.202,0.005394,0.046163,0.116352,0.014743,0.007122,0.238206,0.123675,0.017090,0.009117,0.168711,0
7,495497.0,2007,-0.154404,9.10,0.105711,1149468,0.008246,36250.311,0.006360,-0.029329,0.159155,0.013264,0.005394,0.046163,0.116352,0.014743,0.007122,0.085013,0
8,346597.0,2008,-0.300506,9.26,0.017582,1157000,0.006553,36604.337,0.009766,-0.154404,0.105711,0.008246,0.006360,-0.029329,0.159155,0.013264,0.005394,-0.045857,0
9,382226.0,2009,0.102797,8.84,-0.045356,1162450,0.004710,36961.229,0.009750,-0.300506,0.017582,0.006553,0.009766,-0.154404,0.105711,0.008246,0.006360,-0.161413,1


In [171]:
logistic_regression("Orange")

Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv
Fold 0 Accuracy: 0.667
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         4
           1       0.00      0.00      0.00         2

    accuracy                           0.67         6
   macro avg       0.33      0.50      0.40         6
weighted avg       0.44      0.67      0.53         6

Fold 1 Accuracy: 0.833
              precision    recall  f1-score   support

           0       0.83      1.00      0.91         5
           1       0.00      0.00      0.00         1

    accuracy                           0.83         6
   macro avg       0.42      0.50      0.45         6
weighted avg       0.69      0.83      0.76         6

Fold 2 Accuracy: 0.500
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         3
           1       0.00      0.00      0.00         3

    accuracy   

,Orange,Year,median_pct_change,Mortgage,Mortgage_pct_change,units,units_pct_change,population,pop_pct_change,median_pct_change_lag1,Mortgage_change_lag1,units_change_lag1,pop_change_lag1,median_pct_change_lag2,Mortgage_change_lag2,units_change_lag2,pop_change_lag2,rolling_pct_avg3,price_change_class
0,322046.0,2000,0.130541,3.90,0.157270,972611,0.000000,33987.977,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1
1,360836.0,2001,0.120449,4.15,0.064103,985022,0.012760,34479.458,0.014460,0.130541,0.157270,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1
2,428313.0,2002,0.187002,4.66,0.122892,993678,0.008788,34871.843,0.011380,0.120449,0.064103,0.012760,0.014460,0.130541,0.157270,0.000000,0.000000,0.000000,1
3,546527.0,2003,0.275999,5.66,0.214592,1005278,0.011674,35253.159,0.010935,0.187002,0.122892,0.008788,0.011380,0.120449,0.064103,0.012760,0.014460,0.145997,1
4,635446.0,2004,0.162698,6.36,0.123675,1014596,0.009269,35574.576,0.009117,0.275999,0.214592,0.011674,0.010935,0.187002,0.122892,0.008788,0.011380,0.194483,1
5,719619.0,2005,0.132463,7.10,0.116352,1023742,0.009014,35827.943,0.007122,0.162698,0.123675,0.009269,0.009117,0.275999,0.214592,0.011674,0.010935,0.208566,1
6,714820.0,2006,-0.006669,8.23,0.159155,1030732,0.006828,36021.202,0.005394,0.132463,0.116352,0.009014,0.007122,0.162698,0.123675,0.009269,0.009117,0.190387,0
7,656441.0,2007,-0.081670,9.10,0.105711,1038419,0.007458,36250.311,0.006360,-0.006669,0.159155,0.006828,0.005394,0.132463,0.116352,0.009014,0.007122,0.096164,0
8,453524.0,2008,-0.309117,9.26,0.017582,1045083,0.006417,36604.337,0.009766,-0.081670,0.105711,0.007458,0.006360,-0.006669,0.159155,0.006828,0.005394,0.014708,0
9,544300.0,2009,0.200157,8.84,-0.045356,1047718,0.002521,36961.229,0.009750,-0.309117,0.017582,0.006417,0.009766,-0.081670,0.105711,0.007458,0.006360,-0.132485,1


In [176]:
logistic_regression("Riverside")


Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv
Fold 0 Accuracy: 0.500
              precision    recall  f1-score   support

           0       0.75      0.60      0.67         5
           1       0.00      0.00      0.00         1

    accuracy                           0.50         6
   macro avg       0.38      0.30      0.33         6
weighted avg       0.62      0.50      0.56         6

Fold 1 Accuracy: 0.667
              precision    recall  f1-score   support

           0       1.00      0.60      0.75         5
           1       0.33      1.00      0.50         1

    accuracy                           0.67         6
   macro avg       0.67      0.80      0.62         6
weighted avg       0.89      0.67      0.71         6

Fold 2 Accuracy: 0.667
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         4
           1       0.00      0.00      0.00         2

    accuracy   

,Riverside,Year,median_pct_change,Mortgage,Mortgage_pct_change,units,units_pct_change,population,pop_pct_change,median_pct_change_lag1,Mortgage_change_lag1,units_change_lag1,pop_change_lag1,median_pct_change_lag2,Mortgage_change_lag2,units_change_lag2,pop_change_lag2,rolling_pct_avg3,price_change_class
0,162607.0,2000,0.192781,3.90,0.157270,588488,0.000000,33987.977,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1
1,183627.0,2001,0.129269,4.15,0.064103,605084,0.028201,34479.458,0.014460,0.192781,0.157270,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1
2,218657.0,2002,0.190767,4.66,0.122892,625450,0.033658,34871.843,0.011380,0.129269,0.064103,0.028201,0.014460,0.192781,0.157270,0.000000,0.000000,0.000000,1
3,277239.0,2003,0.267917,5.66,0.214592,649730,0.038820,35253.159,0.010935,0.190767,0.122892,0.033658,0.011380,0.129269,0.064103,0.028201,0.014460,0.170939,1
4,364932.0,2004,0.316308,6.36,0.123675,681618,0.049079,35574.576,0.009117,0.267917,0.214592,0.038820,0.010935,0.190767,0.122892,0.033658,0.011380,0.195984,1
5,411329.0,2005,0.127139,7.10,0.116352,714686,0.048514,35827.943,0.007122,0.316308,0.123675,0.049079,0.009117,0.267917,0.214592,0.038820,0.010935,0.258331,1
6,427333.0,2006,0.038908,8.23,0.159155,750829,0.050572,36021.202,0.005394,0.127139,0.116352,0.048514,0.007122,0.316308,0.123675,0.049079,0.009117,0.237121,0
7,345597.0,2007,-0.191270,9.10,0.105711,776659,0.034402,36250.311,0.006360,0.038908,0.159155,0.050572,0.005394,0.127139,0.116352,0.048514,0.007122,0.160785,0
8,201751.0,2008,-0.416225,9.26,0.017582,790064,0.017260,36604.337,0.009766,-0.191270,0.105711,0.034402,0.006360,0.038908,0.159155,0.050572,0.005394,-0.008408,0
9,191344.0,2009,-0.051583,8.84,-0.045356,796844,0.008582,36961.229,0.009750,-0.416225,0.017582,0.017260,0.009766,-0.191270,0.105711,0.034402,0.006360,-0.189529,0


In [174]:
logistic_regression("San Bernardino")


Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv
Fold 0 Accuracy: 0.167
              precision    recall  f1-score   support

           0       0.50      0.20      0.29         5
           1       0.00      0.00      0.00         1

    accuracy                           0.17         6
   macro avg       0.25      0.10      0.14         6
weighted avg       0.42      0.17      0.24         6

Fold 1 Accuracy: 0.333
              precision    recall  f1-score   support

           0       0.50      0.25      0.33         4
           1       0.25      0.50      0.33         2

    accuracy                           0.33         6
   macro avg       0.38      0.38      0.33         6
weighted avg       0.42      0.33      0.33         6

Fold 2 Accuracy: 0.667
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         4
           1       0.00      0.00      0.00         2

    accuracy   

,San Bernardino,Year,median_pct_change,Mortgage,Mortgage_pct_change,units,units_pct_change,population,pop_pct_change,median_pct_change_lag1,Mortgage_change_lag1,units_change_lag1,pop_change_lag1,median_pct_change_lag2,Mortgage_change_lag2,units_change_lag2,pop_change_lag2,rolling_pct_avg3,price_change_class
0,118965.0,2000,0.044258,3.90,0.157270,602918,0.000000,33987.977,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0
1,137889.0,2001,0.159072,4.15,0.064103,609501,0.010919,34479.458,0.014460,0.044258,0.157270,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1
2,153868.0,2002,0.115883,4.66,0.122892,618334,0.014492,34871.843,0.011380,0.159072,0.064103,0.010919,0.014460,0.044258,0.157270,0.000000,0.000000,0.000000,1
3,188974.0,2003,0.228157,5.66,0.214592,628572,0.016557,35253.159,0.010935,0.115883,0.122892,0.014492,0.011380,0.159072,0.064103,0.010919,0.014460,0.106404,1
4,261724.0,2004,0.384974,6.36,0.123675,640584,0.019110,35574.576,0.009117,0.228157,0.214592,0.016557,0.010935,0.115883,0.122892,0.014492,0.011380,0.167704,1
5,334700.0,2005,0.278828,7.10,0.116352,658704,0.028287,35827.943,0.007122,0.384974,0.123675,0.019110,0.009117,0.228157,0.214592,0.016557,0.010935,0.243004,1
6,335612.0,2006,0.002725,8.23,0.159155,675259,0.025133,36021.202,0.005394,0.278828,0.116352,0.028287,0.007122,0.384974,0.123675,0.019110,0.009117,0.297319,0
7,258437.0,2007,-0.229953,9.10,0.105711,688069,0.018970,36250.311,0.006360,0.002725,0.159155,0.025133,0.005394,0.278828,0.116352,0.028287,0.007122,0.222176,0
8,156478.0,2008,-0.394522,9.26,0.017582,695367,0.010606,36604.337,0.009766,-0.229953,0.105711,0.018970,0.006360,0.002725,0.159155,0.025133,0.005394,0.017200,0
9,135938.0,2009,-0.131264,8.84,-0.045356,698183,0.004050,36961.229,0.009750,-0.394522,0.017582,0.010606,0.009766,-0.229953,0.105711,0.018970,0.006360,-0.207250,0


In [175]:
logistic_regression("Ventura")


Looking for file at: /home/alvaro/DataspellProjects/CA_Housing_Analysis/data/population_clean.csv
Fold 0 Accuracy: 0.667
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         4
           1       0.00      0.00      0.00         2

    accuracy                           0.67         6
   macro avg       0.33      0.50      0.40         6
weighted avg       0.44      0.67      0.53         6

Fold 1 Accuracy: 0.667
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         4
           1       0.00      0.00      0.00         2

    accuracy                           0.67         6
   macro avg       0.33      0.50      0.40         6
weighted avg       0.44      0.67      0.53         6

Fold 2 Accuracy: 0.667
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         4
           1       0.00      0.00      0.00         2

    accuracy   

,Ventura,Year,median_pct_change,Mortgage,Mortgage_pct_change,units,units_pct_change,population,pop_pct_change,median_pct_change_lag1,Mortgage_change_lag1,units_change_lag1,pop_change_lag1,median_pct_change_lag2,Mortgage_change_lag2,units_change_lag2,pop_change_lag2,rolling_pct_avg3,price_change_class
0,298450.0,2000,0.121179,3.90,0.157270,252796,0.000000,33987.977,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1
1,317836.0,2001,0.064956,4.15,0.064103,257075,0.016927,34479.458,0.014460,0.121179,0.157270,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0
2,400407.0,2002,0.259791,4.66,0.122892,260877,0.014789,34871.843,0.011380,0.064956,0.064103,0.016927,0.014460,0.121179,0.157270,0.000000,0.000000,0.000000,1
3,488695.0,2003,0.220496,5.66,0.214592,263779,0.011124,35253.159,0.010935,0.259791,0.122892,0.014789,0.011380,0.064956,0.064103,0.016927,0.014460,0.148642,1
4,612460.0,2004,0.253256,6.36,0.123675,267659,0.014709,35574.576,0.009117,0.220496,0.214592,0.011124,0.010935,0.259791,0.122892,0.014789,0.011380,0.181747,1
5,675675.0,2005,0.103215,7.10,0.116352,270687,0.011313,35827.943,0.007122,0.253256,0.123675,0.014709,0.009117,0.220496,0.214592,0.011124,0.010935,0.244514,1
6,670833.0,2006,-0.007166,8.23,0.159155,275596,0.018135,36021.202,0.005394,0.103215,0.116352,0.011313,0.007122,0.253256,0.123675,0.014709,0.009117,0.192322,0
7,604729.0,2007,-0.098540,9.10,0.105711,278150,0.009267,36250.311,0.006360,-0.007166,0.159155,0.018135,0.005394,0.103215,0.116352,0.011313,0.007122,0.116435,0
8,370754.0,2008,-0.386909,9.26,0.017582,280033,0.006770,36604.337,0.009766,-0.098540,0.105711,0.009267,0.006360,-0.007166,0.159155,0.018135,0.005394,-0.000830,0
9,427894.0,2009,0.154118,8.84,-0.045356,281209,0.004200,36961.229,0.009750,-0.386909,0.017582,0.006770,0.009766,-0.098540,0.105711,0.009267,0.006360,-0.164205,1
